# Feed Ranking — TF two-tower + LinUCB bandit (engagement)

Ranks `Post` objects for the for-you tab (`_rank_for_you` in
`apps/feed/views.py`). Real engagement comes from the Django command
`export_ai_training_data` -> `../data/user/engagement.csv`. When that file is
absent the notebook falls back to a **deterministic synthetic** engagement table
(demo path) so the full pipeline runs end-to-end on any checkout.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


In [ ]:
# Load real engagement (export_ai_training_data) or synthesize for the demo
import os
import numpy as np
import pandas as pd
from pathlib import Path

eng_path = Path('../data/user/engagement.csv')
if eng_path.exists():
    eng = pd.read_csv(eng_path)
    print('REAL engagement rows:', len(eng), '| cols:', list(eng.columns))
else:
    rng = np.random.default_rng(7)
    n_users, n_posts, n_events = 300, 800, 60_000
    user_lat = rng.normal(size=(n_users, 16))
    post_lat = rng.normal(size=(n_posts, 16))
    u = rng.integers(0, n_users, size=n_events)
    p = rng.integers(0, n_posts, size=n_events)
    prob = 1 / (1 + np.exp(-(np.einsum('nk,nk->n', user_lat[u], post_lat[p]) + rng.normal(0, 1.2, n_events))))
    eng = pd.DataFrame({'user_id': u, 'post_id': p,
                        'clicked': rng.random(n_events) < prob})
    print('SYNTHETIC engagement rows:', len(eng),
          '| click rate:', round(eng['clicked'].mean(), 3))
    print('(real data: python manage.py export_ai_training_data -> ../data/user/)')
print(eng.head(3))

In [ ]:
# Two-tower over (user, post) click pairs
import tensorflow as tf

u_cat = eng['user_id'].astype('category')
p_cat = eng['post_id'].astype('category')
eng['u'] = u_cat.cat.codes.values
eng['p'] = p_cat.cat.codes.values
n_users, n_posts = u_cat.cat.categories.size, p_cat.cat.categories.size

rng = np.random.default_rng(1)
pos = eng[eng['clicked'] == 1]
users = np.concatenate([pos['u'], pos['u']]).astype(np.int64)
posts = np.concatenate([pos['p'], rng.integers(0, n_posts, size=len(pos))]).astype(np.int64)
labels = np.concatenate([np.ones(len(pos)), np.zeros(len(pos))]).astype(np.float32)
idx = rng.permutation(len(users))
users, posts, labels = users[idx], posts[idx], labels[idx]
split = int(0.85 * len(users))

from tf_utils import build_two_tower
m = build_two_tower(embed_dim=32, n_users=n_users, n_items=n_posts)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
EPOCHS = {'smoke': 1, 'demo': 4, 'full': 8}[SCALE]
m.fit([users[:split], posts[:split]], labels[:split], epochs=EPOCHS, batch_size=1024,
      validation_data=([users[split:], posts[split:]], labels[split:]), verbose=1)
post_emb = m.layers[3].get_weights()[0]

In [ ]:
# LinUCB sim: pick alpha (exploration) with best cumulative CTR on held-out users
import numpy as np

rng = np.random.default_rng(2)
TEST_USERS, ARMS, STEPS = 40, 60, 300
u_test = rng.integers(0, n_users, size=TEST_USERS)
user_emb_all = m.layers[2].get_weights()[0]       # (n_users, 32) latent interests
arm_ids = rng.integers(0, n_posts, size=ARMS)
Xarm = post_emb[arm_ids]                          # (ARMS, 32) bandit features

def linucb_ctr(alpha, user_emb, steps=STEPS):
    K = len(user_emb)
    A = [np.eye(32) + 1e-2 for _ in range(K)]
    b = [np.zeros(32) for _ in range(K)]
    total = np.zeros(K)
    for _ in range(steps):
        for k, ue in enumerate(user_emb):
            Ainv = np.linalg.inv(A[k])
            theta = Ainv @ b[k]
            mean = Xarm @ theta
            ucb = mean + alpha * np.sqrt(np.einsum('nd,de,ne->n', Xarm, Ainv, Xarm))
            arm = int(np.argmax(ucb))
            reward = float(rng.random() < 1 / (1 + np.exp(-float(ue @ Xarm[arm]))))
            total[k] += reward
            A[k] += np.outer(Xarm[arm], Xarm[arm])
            b[k] += reward * Xarm[arm]
    return total / steps

results = {}
for alpha in (0.0, 0.1, 0.5, 1.0, 2.0):
    results[alpha] = float(linucb_ctr(alpha, user_emb_all[u_test]).mean())
best_alpha = max(results, key=results.get)
print('LinUCB CTR by alpha:', {round(k, 1): round(v, 4) for k, v in results.items()})
print('best alpha:', best_alpha)

### Export contract (consumed by the AI service)

The cells below write `../models/feed_ranker.onnx` and its dynamic-INT8 quantized copy
`feed_ranker_int8.onnx`. `app/ml/serving.py::load_preferred('feed_ranker')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [ ]:
# Export feed ranker ONNX (+ INT8)
from pathlib import Path
import tensorflow as tf
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

emb_dim = m.layers[2].output.shape[-1]
tinp = tf.keras.Input(shape=(1,), dtype='int64')
tx = tf.keras.layers.Embedding(n_users, emb_dim,
                               weights=[m.layers[2].get_weights()[0]])(tinp)
tower = tf.keras.Model(tinp, tf.keras.layers.Flatten()(tx))
onnx = export_keras_onnx(tower, Path('../models'), 'feed_ranker', '1.0.0',
                         input_signature=[tf.TensorSpec((None, 1), tf.int64, name='user_id')])
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'feed_ranker', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'linucb_alpha': best_alpha,
                        'linucb_ctr': round(results[best_alpha], 4)}})
print('exported', q)